In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import re

# --------------------------------------------------
# 1. Read the Bronze table
# --------------------------------------------------

df = spark.table("workforce_analytics.bronze.fact_work_raw")

business_columns = [
    "work_id",
    "people_id",
    "county_id",
    "status",
    "cancellation_id",
    "scheduled_start",
    "scheduled_end"
]

# --------------------------------------------------
# 2. Prepare data types and snapshot dates
# --------------------------------------------------

prepared = (
    df
    .withColumn(
        "snapshot_date",
        F.to_date(
            F.regexp_extract(
                F.col("_source_file"),
                r"(\d{2}_\d{2}_\d{4})",
                1
            ),
            "dd_MM_yyyy"
        )
    )
    .withColumn("work_id", F.col("work_id").cast("long"))
    .withColumn("people_id", F.col("people_id").cast("long"))
    .withColumn("county_id", F.col("county_id").cast("long"))
    .withColumn(
        "cancellation_id",
        F.col("cancellation_id").cast("long")
    )
    .withColumn("status", F.trim(F.col("status")))
    .withColumn(
        "scheduled_start",
        F.to_timestamp("scheduled_start")
    )
    .withColumn(
        "scheduled_end",
        F.to_timestamp("scheduled_end")
    )
)

# --------------------------------------------------
# 3. Get files in chronological order
# --------------------------------------------------

files = [
    row["_source_file"]
    for row in prepared.select("_source_file").distinct().collect()
]

files = sorted(
    files,
    key=lambda x: datetime.strptime(
        re.search(r"(\d{2}_\d{2}_\d{4})", x).group(1),
        "%d_%m_%Y"
    )
)

# Columns retained in the integrated state
state_columns = business_columns + [
    "_source_file",
    "_ingested_at"
]

state = None
all_snapshots = []

# --------------------------------------------------
# 4. Process the full load and subsequent deltas
# --------------------------------------------------

for index, file in enumerate(files):

    print(f"Processing: {file.split('/')[-1]}")

    current = (
        prepared
        .filter(F.col("_source_file") == file)
        .select(
            *state_columns
        )
        # Remove exact duplicate business records
        .dropDuplicates(business_columns)
    )

    # Check that each file has no conflicting duplicate IDs
    duplicate_ids = (
        current.groupBy("work_id")
        .count()
        .filter(F.col("count") > 1)
    )

    if duplicate_ids.limit(1).count() > 0:
        raise ValueError(
            f"Conflicting duplicate work IDs found in {file}"
        )

    snapshot_date = (
        prepared
        .filter(F.col("_source_file") == file)
        .select("snapshot_date")
        .first()["snapshot_date"]
    )

    if index == 0:

        # First file is the initial full load
        state = current

    else:

        # Identify IDs present in this delta
        changed_or_new_ids = current.select("work_id").distinct()

        # Keep previous records whose IDs are not in this delta
        unchanged_previous = state.join(
            changed_or_new_ids,
            on="work_id",
            how="left_anti"
        )

        # Replace old versions and add new records
        state = unchanged_previous.unionByName(current)

    # Save the complete state as it existed on this date
    daily_snapshot = state.withColumn(
        "snapshot_date",
        F.lit(snapshot_date).cast("date")
    )

    all_snapshots.append(daily_snapshot)

    print("Snapshot date:", snapshot_date)
    print("Records in snapshot:", state.count())

# --------------------------------------------------
# 5. Combine all daily snapshots
# --------------------------------------------------

final_df = all_snapshots[0]

for snapshot in all_snapshots[1:]:
    final_df = final_df.unionByName(snapshot)

# --------------------------------------------------
# 6. Save to a NEW Silver test table
# --------------------------------------------------

(
    final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workforce_analytics.silver.fact_work_snapshot_test"
    )
)

print("Fact Work snapshots created successfully.")

In [0]:
test_df = spark.table(
    "workforce_analytics.silver.fact_work_snapshot_test"
)

# Rows in each complete daily snapshot
display(
    test_df.groupBy("snapshot_date")
    .count()
    .orderBy("snapshot_date")
)

# Check for duplicate work IDs within each snapshot
display(
    test_df.groupBy("snapshot_date", "work_id")
    .count()
    .filter(F.col("count") > 1)
)

# Check the final snapshot
display(
    test_df.filter(
        F.col("snapshot_date") == "2026-04-05"
    ).limit(10)
)

In [0]:
from pyspark.sql import functions as F

test_df = spark.table(
    "workforce_analytics.silver.fact_work_snapshot_test"
)

print("1. Duplicate work IDs per snapshot")

display(
    test_df.groupBy("snapshot_date", "work_id")
    .count()
    .filter(F.col("count") > 1)
)

print("2. Invalid scheduled time ranges")

display(
    test_df.filter(
        F.col("scheduled_start").isNull() |
        F.col("scheduled_end").isNull() |
        (F.col("scheduled_end") <= F.col("scheduled_start"))
    )
)

print("3. Missing business IDs")

display(
    test_df.filter(F.col("work_id").isNull())
)